In [15]:
from pathlib import Path
import os

if Path.cwd().name != "onnx":
    os.chdir("onnx")

print(Path.cwd())

/workspaces/dtcllm-introduction-to-rag/onnx


# Download ONNX model Xenova/all-MiniLM-L6-v2

In [2]:
!source .venv/bin/activate && uv run python download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


# Q1. Embedding a query

In [24]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)

print("First value v1[0]:", v1[0])

First value v1[0]: -0.020582036807885073


In [17]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Sample check

In [18]:
print(f"Number of documents: {len(documents)}")
if documents:
    print(f"First document: {documents[0]}")

Number of documents: 72
First document: {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most comm

# Q2. Cosine similarity

In [20]:
filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = next(doc for doc in documents if doc['filename'] == filename)
print(f"Found document with filename: {doc['filename']}")
print(f"Content: {doc['content']}")


Found document with filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Content: # Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over every document, so it takes a minute on our dataset.
Keeping everything in memory is fine here, but a larger dataset would
need too much space.

The third problem is brute-force search. For every query we compare the
query vector against every single document. With 1,000 documents this is
fine, probably even faster than anything smarter. But as the dataset
grows past 10,000 or so, it gets slow, and we'll want

In [26]:
v2 = embed.encode(doc['content'])
score = v1.dot(v2)
print(f"Similarity score between query and document: {score}")


Similarity score between query and document: 0.3610702803026059


# Q3. Chunking and search by hand

In [51]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

Embed in batches

In [52]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 8
X = []

chunk_contents = [chunk['content'] for chunk in chunks]
for start in tqdm(range(0, len(chunk_contents), batch_size)):
    batch = chunk_contents[start:start + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
X.shape

  0%|          | 0/37 [00:00<?, ?it/s]

(295, 384)

In [53]:
scores = X.dot(v1)


Find the highest score document and the corresponding filename

In [54]:
idx = np.argmax(scores)
best_score = scores[idx]
print(f"Best chunk score: {best_score}")

best_chunk = chunks[idx]
print(f"Best chunk index: {idx}")
print(f"Best chunk filename: {best_chunk['filename']}")


Best chunk score: 0.648901732433228
Best chunk index: 94
Best chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


# Q4. Vector search with minsearch

In [56]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [58]:
q4 = "What metric do we use to evaluate a search engine?"
v4 = embed.encode(q4)

vindex.search(v4, num_results=3)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

# Q5. Text search vs vector search

In [59]:
q5 = "How do I store vectors in PostgreSQL?"
v5 = embed.encode(q5)

In [74]:
from minsearch import Index
index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename", "start"],
)
index.fit(chunks)
text_results = index.search(
    q5,
    boost_dict={"filename": 0.5, "content": 2.0}, 
    num_results=5
)

text_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [75]:
vector_results = vindex.search(v5, num_results=5)
vector_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [76]:
text_search_filenames = {result['filename'] for result in text_results}
vector_search_filenames = {result['filename'] for result in vector_results}
only_in_vector_search = vector_search_filenames - text_search_filenames
print(f"Filenames only in vector search results: {only_in_vector_search}")

Filenames only in vector search results: {'02-vector-search/lessons/08-pgvector.md'}


# Q6 Hybird search

Reciprocal Rank Fusion

In [67]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [73]:
q6 = "How do I give the model access to tools?"
v6 = embed.encode(q6)
num_results = 20

text_results = index.search(
    q6,
    boost_dict={"filename": 0.5, "content": 2.0}, 
    num_results=num_results
)

vector_results = vindex.search(v6, num_results=num_results)

results = rrf([vector_results, text_results])
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 